In [ ]:
!pip install sentence-transformers faiss-cpu

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np


knowledge_base = [
    "이 영화는 2023년 개봉한 한국 액션 블록버스터로 관객 800만 명을 돌파했다.",
    "주연 배우의 연기력이 뛰어나다는 평이 많으며 CG 퀄리티가 훌륭하다.",
    "스토리가 진부하고 예측 가능하다는 부정적인 의견도 존재한다.",
    "OST가 매우 감동적이며 엔딩 크레딧까지 자리를 뜨지 못하게 만든다.",
    "러닝타임이 2시간 30분으로 다소 길지만 지루하지 않다는 반응이다.",
    "아이맥스 상영관에서 보면 몰입감이 극대화된다는 관람객 후기가 많다."
]


embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
kb_embeddings = embedder.encode(knowledge_base, convert_to_numpy=True)

dimension = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(kb_embeddings)


classifier_pipeline = pipeline("sentiment-analysis", model="snunlp/KR-FinBert-SC")


def retrieve(query, top_k=2):
    query_vec = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, top_k)
    return [knowledge_base[i] for i in indices[0]]


def rag_sentiment_pipeline(query):
    print(f"[질문]: {query}")


    retrieved = retrieve(query, top_k=2)
    context = " ".join(retrieved)
    print(f"[검색된 문서]: {context}")


    augmented_input = f"{context} {query}"


    result = classifier_pipeline(augmented_input)[0]


    label = result['label']
    score = result['score']
    print(f"[감정 예측]: {label} (확률: {score:.2f})\n")
    print("-" * 60)


queries = [
    "이 영화 배우 연기가 어때요?",
    "스토리는 어떤가요?",
    "아이맥스로 볼 만한가요?"
]

for q in queries:
    rag_sentiment_pipeline(q)